# 🧠 Project 8 – Agentic Customer Analytics Assistant  
## Notebook 04 – LangGraph Agent Workflow

In this notebook, we build an advanced agent using **LangGraph**.

The agent:

- uses the same tools as the basic agent (data overview, return analysis, prediction)
- runs in a graph loop: model → tools → model → answer
- is powered by **Groq (llama-3.1-8b-instant)** via LangChain

In [1]:
import os, sys
from typing import Dict, Any, List, TypedDict

import pandas as pd
import joblib

from dotenv import load_dotenv

from langchain_core.tools import tool
from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    AIMessage,
    ToolMessage,
    BaseMessage,
)

from langchain_groq import ChatGroq

from langgraph.graph import StateGraph, END

# Load .env for GROQ_API_KEY if you use it
load_dotenv()

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")

from src.data_tools import df
from src.agent_langgraph import run_graph_agent

PROJECT_ROOT, DATA_DIR, MODELS_DIR

('C:\\Users\\ynalq\\Projects\\Data-portfolio\\project8-agentic-customer-analytics-assistant',
 'C:\\Users\\ynalq\\Projects\\Data-portfolio\\project8-agentic-customer-analytics-assistant\\data',
 'C:\\Users\\ynalq\\Projects\\Data-portfolio\\project8-agentic-customer-analytics-assistant\\models')

In [2]:
run_graph_agent("Give me a short overview of the dataset: rows, columns and key numeric stats.")

'The dataset contains 2000 rows and 31 columns. The key numeric statistics include:\n\n- Mean CustomerID: 1609.064\n- Mean Age: 44.4875\n- Mean CreditScore: 690.1265\n- Mean MonthlyIncome: 5589.697\n- Mean Cost: 135.816\n- Mean Price: 210.9215\n- Mean Quantity: 3.5275\n- Mean Revenue: 761.2555\n- Mean Margin: 271.3145\n- Mean Recency: 902.4415\n- Mean Frequency: 1.729\n- Mean CustomerTotalRevenue: 1559.367\n- Mean CustomerAvgRevenue: 761.2555\n\nThe dataset also includes information about customer demographics, order details, and return rates.'

In [3]:
run_graph_agent("Which product category has the highest return rate and what does that mean for the business?")

'The product category with the highest return rate is fashion, with a return rate of 15.27%. This means that nearly 1 in 7 fashion products sold are returned, which could be a significant issue for the business. It may be worth investigating the specific products within the fashion category that have the highest return rates, such as shoes and dresses, to identify potential issues with sizing, quality, or customer expectations.\n\nThe business may want to consider implementing strategies to reduce returns in the fashion category, such as improving product descriptions, offering more flexible sizing options, or providing better customer support. By addressing the root causes of returns, the business can reduce waste, improve customer satisfaction, and increase revenue.'

In [4]:
def build_order_features_from_row(idx: int = 0) -> Dict[str, Any]:
    row = df.iloc[idx]

    return {
        "CustomerID": row["CustomerID"],
        "Gender": row["Gender"],
        "Age": int(row["Age"]),
        "CreditScore": float(row["CreditScore"]),
        "MonthlyIncome": float(row["MonthlyIncome"]),
        "Country": row["Country"],
        "State": row["State"],
        "City": row["City"],
        "Category": row["Category"],
        "Product": row["Product"],
        "Cost": float(row["Cost"]),
        "Price": float(row["Price"]),
        "Quantity": int(row["Quantity"]),
        "CampaignSchema": row["CampaignSchema"],
        "PaymentMethod": row["PaymentMethod"],
        "Revenue": float(row["Revenue"]),
        "Margin": float(row["Margin"]),
        "Recency": float(row.get("Recency", 0.0)),
        "Frequency": float(row.get("Frequency", 0.0)),
        "CustomerTotalRevenue": float(row.get("CustomerTotalRevenue", 0.0)),
        "CustomerAvgRevenue": float(row.get("CustomerAvgRevenue", 0.0)),
        "CategoryReturnRate": float(row.get("CategoryReturnRate", 0.0)),
        "ReturnFlag": int(row.get("ReturnFlag", 0)),
    }

In [5]:
example_order = build_order_features_from_row(0)

run_graph_agent(
    "Estimate the return risk for this order and explain it in simple terms. "
    "Only use the model as a rough indicator, and mention its limitations "
    f"Order features: {example_order}"
)

"The model is unable to make a prediction because the 'CategoryReturnRate' feature is missing from the order features. This feature is likely used as an input to the model, and its absence prevents the model from making a prediction.\n\nTo make a prediction, you would need to include the 'CategoryReturnRate' feature in the order features. However, since this feature is a category-level return rate, it's likely that it's not available for individual orders. In that case, you might need to use a different approach or model that doesn't rely on this feature.\n\nIn this case, I would recommend using a different model or approach that doesn't rely on the 'CategoryReturnRate' feature. Alternatively, you could try to estimate the return rate for the category based on other features or data.\n\nLet me know if you'd like to explore other options or approaches."